# MiniCheck faithfulness evaluation — PILOT02

Runs the frozen `flan-t5-large` MiniCheck model on the 118 reviewed sentence-level claims from the three blinded GPT reproducibility summaries.

Before running: choose **Runtime → Change runtime type → T4 GPU**. Then run cells in order. Upload only `PILOT02.txt` and `claim_candidates.csv` when prompted.

In [ ]:
# Verify that Colab supplied a GPU.
import subprocess
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError('No GPU detected. Choose Runtime > Change runtime type > T4 GPU, then reconnect.')
print(gpu.stdout.splitlines()[0])
print('GPU detected.')

In [ ]:
# Install the official MiniCheck implementation. Runtime restart is normally unnecessary.
%pip install -q "minicheck @ git+https://github.com/Liyan06/MiniCheck.git@main"

In [ ]:
# Upload the cleaned policy and frozen claims. Do not upload API keys or the blinding key.
from google.colab import files
uploaded = files.upload()
required = {'PILOT02.txt', 'claim_candidates.csv'}
missing = required - set(uploaded)
if missing:
    raise FileNotFoundError(f'Missing required upload(s): {sorted(missing)}')
print('Required files received.')

In [ ]:
# Validate the frozen inputs before model loading.
import hashlib
import pandas as pd
from pathlib import Path

policy_path = Path('PILOT02.txt')
claims_path = Path('claim_candidates.csv')
policy = policy_path.read_text(encoding='utf-8')
claims = pd.read_csv(claims_path, dtype=str).fillna('')
expected_columns = {'blind_id', 'claim_id', 'claim_text', 'include', 'review_notes'}
if not expected_columns.issubset(claims.columns):
    raise ValueError(f'Claim file is missing columns: {sorted(expected_columns - set(claims.columns))}')
approved = claims[claims['include'].str.strip().str.lower().eq('yes')].copy()
if len(approved) != 118:
    raise ValueError(f'Expected 118 included claims, found {len(approved)}. Stop and verify the frozen file.')
if approved['claim_id'].duplicated().any():
    raise ValueError('Duplicate claim IDs detected.')
expected_blind_ids = {'S014', 'S015', 'S027'}
if set(approved['blind_id']) != expected_blind_ids:
    raise ValueError(f'Unexpected blind IDs: {sorted(set(approved["blind_id"]))}')
if not policy.strip():
    raise ValueError('PILOT02.txt is empty.')
print('Approved claims:', len(approved))
print('Claims per blind ID:', approved.groupby('blind_id').size().to_dict())
print('Policy words:', len(policy.split()))
print('Policy SHA-256:', hashlib.sha256(policy.encode('utf-8')).hexdigest())
print('Claims SHA-256:', hashlib.sha256(claims_path.read_bytes()).hexdigest())

In [ ]:
# Load the frozen MiniCheck model. The first run downloads model weights.
import torch
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from minicheck.minicheck import MiniCheck

if not torch.cuda.is_available():
    raise RuntimeError('PyTorch cannot access the GPU. Reconnect to a GPU runtime.')
MODEL_ID = 'flan-t5-large'
scorer = MiniCheck(model_name=MODEL_ID, cache_dir='/content/minicheck_ckpts')
print('Loaded', MODEL_ID, 'on', torch.cuda.get_device_name(0))

In [ ]:
# Score in small batches and checkpoint after each batch. Safe to rerun.
from datetime import datetime, timezone
from pathlib import Path

OUTPUT = Path('minicheck_claim_scores.csv')
FIELDS = ['blind_id', 'claim_id', 'model_id', 'predicted_supported', 'support_probability', 'timestamp_utc', 'status', 'error']
if OUTPUT.exists():
    existing = pd.read_csv(OUTPUT, dtype=str).fillna('')
    completed = set(existing.loc[existing['status'].eq('success'), 'claim_id'])
    result_rows = existing.to_dict('records')
else:
    completed, result_rows = set(), []

pending = approved[~approved['claim_id'].isin(completed)].copy()
BATCH_SIZE = 4
print(f'Already complete: {len(completed)}; pending: {len(pending)}')
for start in range(0, len(pending), BATCH_SIZE):
    batch = pending.iloc[start:start + BATCH_SIZE]
    try:
        labels, probabilities, _, _ = scorer.score(
            docs=[policy] * len(batch),
            claims=batch['claim_text'].tolist(),
        )
        for (_, row), label, probability in zip(batch.iterrows(), labels, probabilities):
            result_rows.append({
                'blind_id': row['blind_id'],
                'claim_id': row['claim_id'],
                'model_id': MODEL_ID,
                'predicted_supported': int(label),
                'support_probability': float(probability),
                'timestamp_utc': datetime.now(timezone.utc).isoformat(),
                'status': 'success',
                'error': '',
            })
    except Exception as exc:
        for _, row in batch.iterrows():
            result_rows.append({
                'blind_id': row['blind_id'], 'claim_id': row['claim_id'], 'model_id': MODEL_ID,
                'predicted_supported': '', 'support_probability': '',
                'timestamp_utc': datetime.now(timezone.utc).isoformat(),
                'status': 'failed', 'error': f'{type(exc).__name__}: {exc}',
            })
        pd.DataFrame(result_rows, columns=FIELDS).to_csv(OUTPUT, index=False)
        raise
    pd.DataFrame(result_rows, columns=FIELDS).to_csv(OUTPUT, index=False)
    print(f'Checkpoint: {min(start + len(batch), len(pending))}/{len(pending)} pending claims processed')
print('Scoring complete.')

In [ ]:
# Final validation and download.
results = pd.read_csv('minicheck_claim_scores.csv', dtype=str).fillna('')
latest = results.drop_duplicates(subset=['blind_id', 'claim_id'], keep='last')
successful = latest[latest['status'].eq('success')].copy()
if len(latest) != 118 or len(successful) != 118:
    raise ValueError(f'Incomplete result: {len(successful)}/118 successful unique claims.')
if set(successful['model_id']) != {'flan-t5-large'}:
    raise ValueError('Unexpected MiniCheck model ID in results.')
successful['support_probability'] = pd.to_numeric(successful['support_probability'], errors='raise')
if not successful['support_probability'].between(0, 1).all():
    raise ValueError('Support probabilities must be between 0 and 1.')
successful = successful.sort_values(['blind_id', 'claim_id'])
successful.to_csv('minicheck_claim_scores.csv', index=False)
print(successful.groupby(['blind_id', 'predicted_supported']).size())
print('Validated: 118/118 successful unique claims.')
files.download('minicheck_claim_scores.csv')

## Save locally

Place the downloaded `minicheck_claim_scores.csv` in:

`pilot_study/data/pilot/evaluations/faithfulness/`

Do not rename model labels into the four human categories. MiniCheck is a binary evaluator.